In [1]:
from transformers import AutoTokenizer, AutoModelForCausalLM
from src.mbe import patch_mbe 
import torch 
from src.gapt import GatedPhaseTransition

# ----- load model -----
model_name = "Qwen/Qwen3-0.6B"
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForCausalLM.from_pretrained(model_name)

# ----- initialize GAPT ----- 
gapt = GatedPhaseTransition(
    p_m=125, # patience plateaued ce loss 
    p_a=75 # patience for plateaued mbe loss 
)

# ----- compute loss ----
num_layers = model.config.num_hidden_layers
per_layer_mbe_mask = torch.zeros(num_layers) # <- require ablation on this mask
per_layer_mbe_mask[1:-1] = 1 
patch_size = 3

inputs = tokenizer("Hello, how are you?", return_tensors="pt")
assert len(inputs["input_ids"][0]) % patch_size == 0, "patch size needs to divide seq len here"

# Request #1. Include 'spike' & 'bottleneck' schedule to replace the 'naive' schedule here
def compute_loss(model, inputs, patch_size: int, per_layer_mbe_mask: torch.Tensor, gapt: GatedPhaseTransition, mbe_comp_mode: str = "spike"):
    outputs = model(**inputs, labels=inputs["input_ids"], output_hidden_states=True, return_dict=True)
    ce_loss = outputs.loss 

    mbe_per_layer = torch.stack([patch_mbe(h, patch_size).float() for h in outputs.hidden_states[1:]])
    masked_mbe = mbe_per_layer * per_layer_mbe_mask
    if mbe_comp_mode == "naive": 
        mbe_loss = (masked_mbe.sum() / per_layer_mbe_mask.sum())
    elif mbe_comp_mode == "bottleneck": 
        gradients = masked_mbe[1:] - masked_mbe[:-1]
        k = max(1, int(len(masked_mbe) * 0.1))
        _, indices = torch.topk(gradients, k, largest=False)
        mbe_loss = masked_mbe[indices + 1].mean()
    elif mbe_comp_mode == "spike": 
        gradients = masked_mbe[1:] - masked_mbe[:-1]
        decay_idx = gradients.argmin()
        mbe_loss = masked_mbe[decay_idx + 1]

    loss = gapt.step(ce_loss, mbe_loss)
    return loss

In [ ]:
from gapt_trainer import GaptTrainer, GaptConfig
from transformers import TrainingArguments
from transformers import AutoTokenizer, AutoModelForCausalLM
from datasets import load_dataset

# ----- load model -----
model_name = "Qwen/Qwen3-0.6B"
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForCausalLM.from_pretrained(model_name)

dataset = load_dataset("gsm8k", "main")

# Formatting function
def format_gsm8k(example):
    return {"text": example["question"] + "\n" + example["answer"]}

# Tokenize
def tokenize_function(examples):
    return tokenizer(examples["text"], padding="max_length", truncation=True, max_length=512)

tokenized_datasets = dataset.map(format_gsm8k).map(tokenize_function, batched=True)


# ---- GAPT config & trainer ----
gapt_config = GaptConfig()
gapt_trainer = GaptTrainer(
    gapt_config = gapt_config,
    model = model,
    train_dataset=tokenized_datasets["train"],
    eval_dataset=tokenized_datasets["test"],
    args = TrainingArguments(
        per_device_train_batch_size=1,
        per_device_eval_batch_size=1,
        num_train_epochs=1,
    )
)

# train 
gapt_trainer.train()

huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)


Map:   0%|          | 0/1319 [00:00<?, ? examples/s]

huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)
huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)
huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)
huggingface/tokenizers: The 

huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)
huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)
huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)


/Users/ksgk/anaconda3/lib/python3.12/site-packages/torch/utils/data/dataloader.py:684: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, then device pinned memory won't be used.
  warnings.warn(warn_msg)


Step,Training Loss
